# 1.3 — Wine Data EDA on the **original** white-wine dataset (baseline)

This notebook performs EDA on the original Wine Dataset (`data/raw/wine_quality_original.csv`, the white-wine data).

**Note:** `wine_quality_original.csv` contains **many duplicate rows**: 937 of its 4,898 rows (about 19%) are exact repeats of another row.
Duplicates over-weight some wines and bias statistics, so this notebook first removes them, saves the result as
`data/interim/wine_quality_original_no_duplicates.csv`, and runs the whole EDA on that deduplicated dataset.

After reviewing this notebook, run the next one: `1.4-grh-wine-data-cleaning.ipynb` that shows different cleaning strategies for a perturbed dataset (artificially introduced naturally occurring defects in raw datasets) derived from the original wine quality data.
`wine_quality_original_no_duplicates.csv` generated in this notebook will be used as the **baseline** for the next notebook, so run this notebook before 1.4.


This notebook was developed with support of Anthropic's Claude Opus 5.5

> **Paths.** All file locations come from `mlops/config.py` (`RAW_DATA_DIR`, `INTERIM_DATA_DIR`, `PROCESSED_DATA_DIR`), so the notebook works no matter which folder Jupyter is started from. Install the project first: `pip install -r requirements.txt` (it installs the `mlops` package in editable mode).

# Setup

In [ ]:
# Loading the libraries (EDA only)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mlops.config import INTERIM_DATA_DIR, RAW_DATA_DIR

In [ ]:
# Setting up tweaks for the visualization
sns.set(style="whitegrid")

# Loading the data

In [ ]:
# Load the original dataset (saved by notebook 1.0 in data/raw)
ORIGINAL_PATH = RAW_DATA_DIR / "wine_quality_original.csv"
NO_DUPLICATES_PATH = INTERIM_DATA_DIR / "wine_quality_original_no_duplicates.csv"

original_df = pd.read_csv(ORIGINAL_PATH)
print(f"Loaded: data/raw/{ORIGINAL_PATH.name}  ({original_df.shape[0]:,} rows, {original_df.shape[1]} columns)")
original_df.head()

## Removing duplicate rows

`wine_quality_original.csv` contains many exact duplicate rows. Before exploring the data we count them, drop them, and save the
deduplicated dataset. The rest of the notebook works on `raw_wine_df`, which from here on is the **deduplicated** data.

In [ ]:
# Count exact duplicate rows in the original file
n_dup_original = original_df.duplicated().sum()
print(f"Duplicated rows in {ORIGINAL_PATH.name}: {n_dup_original:,} ({100 * n_dup_original / len(original_df):.2f}%)")

# Remove them (the first occurrence of each row is kept) and save the baseline for notebook 1.4
raw_wine_df = original_df.drop_duplicates().reset_index(drop=True)
raw_wine_df.to_csv(NO_DUPLICATES_PATH, index=False)

print(f"Rows after removing duplicates: {len(raw_wine_df):,}")
print(f"Saved: data/interim/{NO_DUPLICATES_PATH.name}")

In [ ]:
raw_wine_df.info()

In [ ]:
raw_wine_df.head().T

In [ ]:
raw_wine_df.describe().T

# Exploratory Data Analysis (EDA)

The goal of this section is to understand the dataset before applying any preprocessing or training a model.

We will review:

1. Dataset structure and data quality.
2. Target variable (`quality`) distribution.
3. Univariate analysis of numerical variables.
4. Skewness and potential outliers.
5. Bivariate relationships between predictors and wine quality.
6. Relationships among predictors.
7. Correlation analysis using Pearson and Spearman coefficients.
8. Multivariate profiles by wine quality.


## Dataset structure and data quality

Before analyzing distributions, verify the shape, data types, missing values, duplicated observations, and the number of unique values per column.


In [ ]:
# Dataset dimensions
print(f"Rows: {raw_wine_df.shape[0]:,}")
print(f"Columns: {raw_wine_df.shape[1]}")

# Compact data-quality summary
data_quality = pd.DataFrame({
    "dtype": raw_wine_df.dtypes,
    "missing": raw_wine_df.isna().sum(),
    "missing_pct": raw_wine_df.isna().mean() * 100,
    "n_unique": raw_wine_df.nunique()
})

data_quality.sort_values("missing_pct", ascending=False)


In [ ]:
# Duplicate observations (expected to be 0: duplicates were removed right after loading)
n_duplicates = raw_wine_df.duplicated().sum()
duplicate_pct = 100 * n_duplicates / len(raw_wine_df)

print(f"Duplicated rows: {n_duplicates:,}")
print(f"Percentage of duplicated rows: {duplicate_pct:.2f}%")


## Target variable analysis: `quality`

Because `quality` is the response variable, inspect both its absolute frequency and relative frequency. This helps identify class imbalance before any train/test split is performed.


In [ ]:
quality_summary = pd.DataFrame({
    "count": raw_wine_df["quality"].value_counts().sort_index(),
    "percentage": raw_wine_df["quality"].value_counts(normalize=True).sort_index() * 100
})

quality_summary


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=raw_wine_df, x="quality")

plt.title("Distribution of Wine Quality")
plt.xlabel("Quality")
plt.ylabel("Number of observations")

for container in ax.containers:
    ax.bar_label(container)

plt.show()


## Univariate analysis

Analyze each numerical feature independently to understand its center, spread, shape, skewness, and potential extreme observations.


In [ ]:
numeric_features = [
    col for col in raw_wine_df.select_dtypes(include=np.number).columns
    if col != "quality"
]

univariate_summary = raw_wine_df[numeric_features].describe().T

univariate_summary["median"] = raw_wine_df[numeric_features].median()
univariate_summary["skewness"] = raw_wine_df[numeric_features].skew()
univariate_summary["kurtosis"] = raw_wine_df[numeric_features].kurt()

univariate_summary[
    ["count", "mean", "median", "std", "min", "25%", "50%", "75%", "max",
     "skewness", "kurtosis"]
].sort_values("skewness", key=lambda x: x.abs(), ascending=False)


In [ ]:
# Distribution + boxplot for every numerical predictor
for column in numeric_features:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(
        data=raw_wine_df,
        x=column,
        kde=True,
        ax=axes[0]
    )
    axes[0].set_title(f"Distribution of {column}")

    sns.boxplot(
        data=raw_wine_df,
        x=column,
        ax=axes[1]
    )
    axes[1].set_title(f"Boxplot of {column}")

    plt.tight_layout()
    plt.show()


### Why use both plots?

The histogram/KDE shows the **shape of the distribution**, while the boxplot makes **dispersion and potential extreme values** easier to identify.

Pay special attention to:

- Long right or left tails.
- Multiple modes.
- Variables concentrated in narrow ranges.
- Very large values relative to the bulk of the data.

An extreme value should not automatically be removed. Be aware that domain expertise should be considered before deciding if those extreme values should be filtered or not.


## Potential outliers using the IQR rule

The IQR rule is useful as a diagnostic tool. It flags observations outside:

\[
[Q1 - 1.5 \times IQR,\; Q3 + 1.5 \times IQR]
\]

These observations are **potential outliers**, not necessarily errors.


In [ ]:
outlier_rows = []

for column in numeric_features:
    q1 = raw_wine_df[column].quantile(0.25)
    q3 = raw_wine_df[column].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (
        (raw_wine_df[column] < lower_bound) |
        (raw_wine_df[column] > upper_bound)
    )

    outlier_rows.append({
        "feature": column,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "n_potential_outliers": int(mask.sum()),
        "outlier_pct": 100 * mask.mean()
    })

outlier_summary = (
    pd.DataFrame(outlier_rows)
      .sort_values("outlier_pct", ascending=False)
)

outlier_summary


A feature can show many IQR outliers simply because its natural distribution is strongly skewed. This distinction matters later when deciding whether transformations such as `log1p`, robust scaling, clipping, or no transformation at all are appropriate.

## Bivariate analysis: predictors vs. wine quality

Now analyze how each physicochemical variable changes across values of `quality`.


In [ ]:
# Descriptive statistics of each feature grouped by wine quality
quality_profiles = raw_wine_df.groupby("quality")[numeric_features].agg(
    ["mean", "median", "std"]
)

quality_profiles


In [ ]:
# Boxplots by quality
for column in numeric_features:
    plt.figure(figsize=(9, 5))
    sns.boxplot(
        data=raw_wine_df,
        x="quality",
        y=column
    )
    plt.title(f"{column} by Wine Quality")
    plt.xlabel("Quality")
    plt.ylabel(column)
    plt.show()


In [ ]:
# Mean feature value by quality.
# This view makes monotonic or non-monotonic trends easier to spot.
for column in numeric_features:
    grouped = (
        raw_wine_df.groupby("quality", as_index=False)[column]
        .mean()
    )

    plt.figure(figsize=(8, 4))
    sns.lineplot(
        data=grouped,
        x="quality",
        y=column,
        marker="o"
    )
    plt.title(f"Mean {column} by Wine Quality")
    plt.xlabel("Quality")
    plt.ylabel(f"Mean {column}")
    plt.show()


## Pairwise relationships among predictors

Strong relationships among predictors may reveal redundant information, common chemical mechanisms, or potential multicollinearity for some model families.


In [ ]:
# Pearson correlation: focuses on linear relationships
pearson_corr = raw_wine_df[numeric_features + ["quality"]].corr(method="pearson")

plt.figure(figsize=(13, 10))
sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Pearson Correlation Matrix")
plt.show()


In [ ]:
# Spearman correlation: based on ranks and useful for monotonic,
# potentially non-linear relationships
spearman_corr = raw_wine_df[numeric_features + ["quality"]].corr(method="spearman")

plt.figure(figsize=(13, 10))
sns.heatmap(
    spearman_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Spearman Correlation Matrix")
plt.show()


In [ ]:
# Compare feature correlations with quality under both measures
quality_correlations = pd.DataFrame({
    "pearson_with_quality": pearson_corr["quality"].drop("quality"),
    "spearman_with_quality": spearman_corr["quality"].drop("quality")
})

quality_correlations["abs_pearson"] = (
    quality_correlations["pearson_with_quality"].abs()
)

quality_correlations.sort_values(
    "abs_pearson",
    ascending=False
).drop(columns="abs_pearson")


### Pearson vs. Spearman

- **Pearson** measures the strength of a linear relationship.
- **Spearman** measures the strength of a monotonic relationship based on ranks.

If a feature has a noticeably stronger Spearman correlation than Pearson correlation, its relationship with quality may be monotonic but not well represented by a straight line.


## Strongly correlated predictor pairs

Instead of visually scanning the complete correlation matrix, we could extract the pairs with the highest absolute correlations.


In [ ]:
feature_corr = raw_wine_df[numeric_features].corr()

upper_triangle = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

strong_pairs = (
    upper_triangle.stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={
        "level_0": "feature_1",
        "level_1": "feature_2"
    })
)

strong_pairs["abs_correlation"] = strong_pairs["correlation"].abs()

strong_pairs.sort_values(
    "abs_correlation",
    ascending=False
).head(15)


## Focused pairwise visualization

A full pairplot with every feature can become difficult to read. Instead, select the features with the strongest absolute association with `quality` and inspect them jointly.


In [ ]:
top_features = (
    quality_correlations["pearson_with_quality"]
    .abs()
    .sort_values(ascending=False)
    .head(4)
    .index
    .tolist()
)

print("Features selected for focused pairplot:", top_features)

sns.pairplot(
    raw_wine_df[top_features + ["quality"]],
    hue="quality",
    corner=True,
    diag_kind="hist"
)

plt.show()


## Standardized multivariate profile by quality

Features are measured on very different scales. Standardizing the group means only for visualization makes it easier to compare how the typical feature changes across quality levels.

This transformation is exploratory and does not modify the raw dataset.


In [ ]:
feature_means_by_quality = (
    raw_wine_df.groupby("quality")[numeric_features]
    .mean()
)

standardized_profiles = (
    feature_means_by_quality - raw_wine_df[numeric_features].mean()
) / raw_wine_df[numeric_features].std()

plt.figure(figsize=(14, 7))
sns.heatmap(
    standardized_profiles,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f"
)

plt.title("Standardized Mean Feature Profile by Wine Quality")
plt.xlabel("Features")
plt.ylabel("Wine quality")
plt.show()


Each value indicates how far the **mean for a quality group** is from the overall feature mean, measured in standard deviations.

- Positive values: above the dataset-wide mean.
- Negative values: below the dataset-wide mean.
- Values near zero: close to the overall mean.


>The purpose of EDA is not to justify a model that has already been selected. It is to identify properties of the data that should inform subsequent decisions.


## Next step: comparison in notebook 1.4

Notebook `1.4-grh-wine-data-cleaning.ipynb` cleans the perturbed file `wine_quality_modified.csv` with two strategies (dropping invalid
rows, and filling invalid values with an estimate), removes duplicates in both, and compares the results against
`wine_quality_original_no_duplicates.csv` generated here. The EDA results of this notebook are the reference for that comparison:

1. **Target distribution.** The class proportions of `quality`.
2. **Univariate summary.** Means, medians and spread; skewness and kurtosis are more sensitive, because they depend on a
   handful of extreme but valid wines.
3. **IQR outliers.** The share of *potential* outliers created by the natural tails of skewed variables.
4. **Correlations with quality.** Which features (e.g. alcohol, density, chlorides) are most associated with quality.
5. **Standardized profiles by quality.** How the typical wine changes across quality levels.